# OPTIONAL - merge manually corrected missing smiles/names - OBS HSBD data only
Uses Opsin to correct some structures names to smiles.

This step can be skipped if manual curation wasn't done.

In [39]:
import sys
from typing import List
import pandas as pd
from pandas import read_csv
import numpy as np
from pathlib import Path
import requests
from tqdm.notebook import tqdm

notebookdir = Path.cwd().parent
sys.path.append(str(notebookdir))  # this way we can import src modules even in different subfolders
from src.rdkit_tools import smiles_standardizer_msc

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [40]:
working_dir = Path.cwd().parent
raw_data_dir = working_dir / "raw_data"
output_dir = working_dir / "processed_data"
hsbd_temporary_dir = output_dir / "missing" / "_hsbd_tmp"

corrected_combo_file = output_dir / "missing" / "missing_names_combo_manual_corrected_ready2merge.csv"
if not corrected_combo_file.exists():
    raise FileNotFoundError(
        f"Corrected combo file not found at {corrected_combo_file}. Please ensure the file exists and is in the correct location."
    )

corrected_combo_names = read_csv(corrected_combo_file, header=0, sep="\t")
corrected_combo_names.dropna(inplace=True)
corrected_combo_names.reset_index(drop=True, inplace=True)
print(f"Loaded {len(corrected_combo_names)} manually corrected entries ready to merge into the main dataframes.")

Loaded 49 manually corrected entries ready to merge into the main dataframes.


In [41]:
air_data = read_csv(hsbd_temporary_dir / "hsbd_air_w_duplicates_for_optional.tmp", header=0, sep="\t")
soil_data = read_csv(hsbd_temporary_dir / "hsbd_soil_w_duplicates_for_optional.tmp", header=0, sep="\t")
water_data = read_csv(hsbd_temporary_dir / "hsbd_water_w_duplicates_for_optional.tmp", header=0, sep="\t")
sediment_data = read_csv(hsbd_temporary_dir / "hsbd_sediment_w_duplicates_for_optional.tmp", header=0, sep="\t")

In [ ]:
def fetch_missing_entries(df: pd.DataFrame, column_name: str = "Canonical_smiles") -> list[str]:
    """Fetch list of compounds with missing entries in specified column.
    Checks for NaN, 'None' (string), and empty string "".
    """
    missing_mask = (
        df[column_name].isna()
        | (df[column_name].astype(str).str.lower() == "nan")
        | (df[column_name].astype(str).str.lower() == "none")
        | (df[column_name] == "")
    )
    missing_entries = df[missing_mask]["Compound"].tolist()
    return missing_entries


def print_missing_smiles_stats(df: pd.DataFrame, missing_df: list, label: str) -> None:
    print(
        f"Extracted {len(df)} {label} entries, with {len(missing_df)} missing SMILES = {len(missing_df) / len(df) * 100:.2f}%."
    )

In [43]:
"""Functions to get SMILES via InChI from OPSIN but clean via rdkit
Note that we do a MSC smiles clean later on again (wasn't availble at this point of writing)
"""


def _smiles_from_inchi(inchi: str) -> str | None:
    from rdkit import Chem

    """Convert InChI string to SMILES using RDKit."""
    mol = Chem.inchi.MolFromInchi(inchi)
    if mol:
        smiles = Chem.MolToSmiles(mol)
        return smiles
    print(f"Could not convert InChI to SMILES: {inchi}")
    return None


def fetch_smiles_via_inchi_from_name(iupac_name: str) -> str | None:
    """Fetch InChI string from IUPAC name using OPSIN API.
    Returns InChI string if found, else None."""
    inchi: str | None = None
    response = requests.get(f"https://opsin.ch.cam.ac.uk/opsin/{iupac_name}.json")
    # Parse the response
    if response.status_code == 200:
        data = response.json()
        inchi = data.get("stdinchi")
        smiles = _smiles_from_inchi(inchi) if inchi else None
        return smiles if smiles else None
    else:
        return None

In [44]:
# get smiles from "manual correct" column
# iterate over rows and update dataframes accordingly
new_smiles_list = []
print(f"Number of names to process: {len(corrected_combo_names)}")
for index, row in tqdm(corrected_combo_names.iterrows(), desc="Processing names"):
    corrected_name = row["manual correct"]
    smiles = fetch_smiles_via_inchi_from_name(corrected_name)
    new_smiles_list.append(smiles)

corrected_combo_names["Canonical_smiles"] = new_smiles_list

Number of names to process: 49


Processing names: 0it [00:00, ?it/s]

In [45]:
# in case it didn't work for some, drop those rows
corrected_combo_names.dropna(subset=["Canonical_smiles"], inplace=True)

In [46]:
# now update the dataframes with the new SMILES
for index, row in tqdm(corrected_combo_names.iterrows(), desc="Updating dataframes with corrected names"):
    original_name = row["original_name"]
    new_smiles = row["Canonical_smiles"]
    air_data.loc[air_data["Compound"] == original_name, "Canonical_smiles"] = new_smiles
    soil_data.loc[soil_data["Compound"] == original_name, "Canonical_smiles"] = new_smiles
    water_data.loc[water_data["Compound"] == original_name, "Canonical_smiles"] = new_smiles
    sediment_data.loc[sediment_data["Compound"] == original_name, "Canonical_smiles"] = new_smiles
    new_name = row["manual correct"]
    air_data.loc[air_data["Compound"] == original_name, "Corrected_names"] = new_name
    soil_data.loc[soil_data["Compound"] == original_name, "Corrected_names"] = new_name
    water_data.loc[water_data["Compound"] == original_name, "Corrected_names"] = new_name
    sediment_data.loc[sediment_data["Compound"] == original_name, "Corrected_names"] = new_name

Updating dataframes with corrected names: 0it [00:00, ?it/s]

In [ ]:
# update the lists of missing SMILES after fetching
missing_smiles_air: list[str] = fetch_missing_entries(air_data)
missing_smiles_soil: list[str] = fetch_missing_entries(soil_data)
missing_smiles_water: list[str] = fetch_missing_entries(water_data)
missing_smiles_sediment: list[str] = fetch_missing_entries(sediment_data)
missing_smiles_lists = [
    missing_smiles_air,
    missing_smiles_soil,
    missing_smiles_water,
    missing_smiles_sediment,
]
# also update the dataframe list
dataframe_list = [air_data, soil_data, water_data, sediment_data]

# print updated missing SMILES stats
labels = ["air", "soil", "water", "sediment"]  # updated labels, but not executed with labels as of yet
for df, ml, label in zip(dataframe_list, missing_smiles_lists, labels):
    print_missing_smiles_stats(df, ml, label)

Extracted 562 air t_half entries, with 15 missing SMILES = 2.67%.
Extracted 693 air t_half entries, with 8 missing SMILES = 1.15%.
Extracted 757 air t_half entries, with 33 missing SMILES = 4.36%.
Extracted 360 air t_half entries, with 7 missing SMILES = 1.94%.


In [48]:
# standardize smiles - copied from database creation notebook, but with some minor changes
df2_dict = {
    "air": air_data,
    "soil": soil_data,
    "water": water_data,
    "sediment": sediment_data,
}
# get all "Canonical_smiles", created Canonical SMILES to ensure all are actually canonical
# this should have been done during earlier data processing, but just to be sure and then not have any issues later on
for df in df2_dict.values():
    print(f"Before standardization: {len(df)}")
    unique_smiles: List[str] = df["Canonical_smiles"].unique().tolist()
    canonical_smiles = {}
    for smi in unique_smiles:
        smi_str = str(smi) if smi is not None else ""
        if smi_str and smi_str.lower() not in ["nan", "none"]:
            canonical_smiles[smi] = smiles_standardizer_msc(smi)[0]
        else:
            canonical_smiles[smi] = None
    df["Canonical_smiles"] = df["Canonical_smiles"].map(canonical_smiles)
    df.dropna(subset=["Canonical_smiles"], inplace=True)  # now we can drop still missing entries
    df.reset_index(drop=True, inplace=True)
    print(f"After standardization: {len(df)}\n")

Before standardization: 562
After standardization: 547

Before standardization: 693
After standardization: 685

Before standardization: 757
After standardization: 724

Before standardization: 360
After standardization: 353



In [49]:
# double check for NaN values in Canonical_smiles column - redundancy check
print("\nChecking for NaN values in Canonical_smiles column:")
print(f"Air data: {air_data['Canonical_smiles'].isna().sum()} NaN values")
print(f"Soil data: {soil_data['Canonical_smiles'].isna().sum()} NaN values")
print(f"Water data: {water_data['Canonical_smiles'].isna().sum()} NaN values")
print(f"Sediment data: {sediment_data['Canonical_smiles'].isna().sum()} NaN values")
air_data.dropna(subset=["Canonical_smiles"], inplace=True)
soil_data.dropna(subset=["Canonical_smiles"], inplace=True)
water_data.dropna(subset=["Canonical_smiles"], inplace=True)
sediment_data.dropna(subset=["Canonical_smiles"], inplace=True)
print(f"Air data: {air_data['Canonical_smiles'].isna().sum()} NaN values")
print(f"Soil data: {soil_data['Canonical_smiles'].isna().sum()} NaN values")
print(f"Water data: {water_data['Canonical_smiles'].isna().sum()} NaN values")
print(f"Sediment data: {sediment_data['Canonical_smiles'].isna().sum()} NaN values")


Checking for NaN values in Canonical_smiles column:
Air data: 0 NaN values
Soil data: 0 NaN values
Water data: 0 NaN values
Sediment data: 0 NaN values
Air data: 0 NaN values
Soil data: 0 NaN values
Water data: 0 NaN values
Sediment data: 0 NaN values


In [50]:
def deduplicate_and_count_sources(df: pd.DataFrame, smiles_col: str = "Canonical_smiles") -> pd.DataFrame:
    counts = df.groupby(smiles_col).size().rename("n_sources")
    df = df.drop_duplicates(subset=smiles_col).merge(counts, on=smiles_col)
    return df


air_before_dedup = len(air_data)
soil_before_dedup = len(soil_data)
water_before_dedup = len(water_data)
sediment_before_dedup = len(sediment_data)
air_data = deduplicate_and_count_sources(air_data).reset_index(drop=True)
soil_data = deduplicate_and_count_sources(soil_data).reset_index(drop=True)
water_data = deduplicate_and_count_sources(water_data).reset_index(drop=True)
sediment_data = deduplicate_and_count_sources(sediment_data).reset_index(drop=True)
print(f"Air data: {air_before_dedup} entries before deduplication, {len(air_data)} after deduplication.")
print(f"Soil data: {soil_before_dedup} entries before deduplication, {len(soil_data)} after deduplication.")
print(f"Water data: {water_before_dedup} entries before deduplication, {len(water_data)} after deduplication.")
print(f"Sediment data: {sediment_before_dedup} entries before deduplication, {len(sediment_data)} after deduplication.")

Air data: 547 entries before deduplication, 308 after deduplication.
Soil data: 685 entries before deduplication, 672 after deduplication.
Water data: 724 entries before deduplication, 673 after deduplication.
Sediment data: 353 entries before deduplication, 347 after deduplication.


In [51]:
# Final save of processed data
air_data.to_csv(output_dir / "hsbd_air_t_half_data_incl_missing.csv", index=False, sep="\t")
soil_data.to_csv(output_dir / "hsbd_soil_t_half_data_incl_missing.csv", index=False, sep="\t")
water_data.to_csv(output_dir / "hsbd_water_t_half_data_incl_missing.csv", index=False, sep="\t")
sediment_data.to_csv(output_dir / "hsbd_sediment_t_half_data_incl_missing.csv", index=False, sep="\t")